# Individual ROI Urine Test Strip Detector
This version allows you to **adjust each of the 14 pads individually**.

### Instructions:
1. Run **Cell 1** to upload your image.
2. Run **Cell 2** to initialize the individual coordinate system.
3. Run **Cell 3** to see the interactive interface.
   - Use the **Dropdown** to select which pad you want to move.
   - Use the **Sliders** to position that specific pad.
   - All other pads will stay in their last saved positions.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from ipywidgets import interact, IntSlider, Dropdown, Button, Output
import ipywidgets as widgets

# 1. Upload the image
print("Please upload an image of the test strip...")
uploaded = files.upload()
if uploaded:
    file_name = list(uploaded.keys())[0]
    img_bgr = cv2.imread(file_name)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape
    print(f"Successfully loaded {file_name} ({w}x{h})")
else:
    print("No file uploaded.")

In [ ]:
# 2. Initialize Individual ROI Coordinates
PARAMETERS = [
    "Urobilinogen", "Bilirubin", "Ketone", "Creatinine", "Blood", 
    "Protein", "Micro Albumin", "Nitrite", "Leukocytes", "Glucose", 
    "Specific Gravity", "pH", "Ascorbate", "Calcium"
]

# Default starting positions (linear stack)
roi_data = {}
for i, name in enumerate(PARAMETERS):
    roi_data[name] = {
        'x': w // 2,
        'y': 50 + (i * 40),
        'size': 30
    }

print("Coordinates initialized for 14 pads.")

In [ ]:
# 3. Individual Adjustment Interface

def update_and_show(selected_param, x, y, size):
    # Update the data for the selected parameter
    roi_data[selected_param]['x'] = x
    roi_data[selected_param]['y'] = y
    roi_data[selected_param]['size'] = size
    
    temp_img = img_rgb.copy()
    results = []

    for i, name in enumerate(PARAMETERS):
        curr_x = roi_data[name]['x']
        curr_y = roi_data[name]['y']
        curr_size = roi_data[name]['size']
        
        y_end = min(curr_y + curr_size, h)
        x_end = min(curr_x + curr_size, w)
        
        # Color: Green for the one you are currently editing, Blue for others
        color = (0, 255, 0) if name == selected_param else (255, 0, 0)
        
        cv2.rectangle(temp_img, (curr_x, curr_y), (x_end, y_end), color, 2)
        cv2.putText(temp_img, str(i+1), (curr_x - 35, curr_y + 15), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # Extract Color
        roi = img_rgb[curr_y:y_end, curr_x:x_end]
        if roi.size > 0:
            avg_color = cv2.mean(roi)[:3]
            results.append((name, avg_color))
        else:
            results.append((name, (0, 0, 0)))

    plt.figure(figsize=(12, 18))
    plt.imshow(temp_img)
    plt.title(f"Adjusting: {selected_param}")
    plt.axis('off')
    plt.show()

    print(f"\n{'Parameter':<20} | {'R':<4} | {'G':<4} | {'B':<4}")
    print("-" * 40)
    for name, color in results:
        print(f"{name:<20} | {int(color[0]):<4} | {int(color[1]):<4} | {int(color[2]):<4}")

# Create the interactive elements
param_dropdown = Dropdown(options=PARAMETERS, description='Target Pad:')
x_slider = IntSlider(min=0, max=w, step=1, value=w//2, description='X Pos')
y_slider = IntSlider(min=0, max=h, step=1, value=50, description='Y Pos')
size_slider = IntSlider(min=5, max=150, step=1, value=30, description='Size')

def on_dropdown_change(change):
    # When user picks a new pad from list, move sliders to its current saved position
    new_name = change['new']
    x_slider.value = roi_data[new_name]['x']
    y_slider.value = roi_data[new_name]['y']
    size_slider.value = roi_data[new_name]['size']

param_dropdown.observe(on_dropdown_change, names='value')

interact(update_and_show, 
         selected_param=param_dropdown,
         x=x_slider,
         y=y_slider,
         size=size_slider)